In [ ]:
# import standard packages you'll need

import sys
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import Audio, display

print(sys.executable)

# ─ add your own additional imports here:
import librosa
import librosa.display
import seaborn as sns
from tqdm.auto import tqdm
import soundfile as sf


In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────────
CORPUS_DIR = Path("data") / "your-genre-corpus"   # ← name of your corpus folder
# ─────────────────────────────────────────────────────────────────────────
SESSION = "T05"
PREVIEW_DIR = CORPUS_DIR / "previews"
STEM_DIR = CORPUS_DIR / "stems"
STEM_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = Path("output") / SESSION      # results of this session
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert CORPUS_DIR.is_dir(), f"Not found: {CORPUS_DIR.resolve()} → check CORPUS_DIR"
CHART_CSV = next(CORPUS_DIR.glob("*.csv"))
print(CHART_CSV)

## Deep-learning based source separation: PyTorch & Demucs

**PyTorch** is a Python library for deep learning (Meta, open source):
- **Tensors** as fundamental data structure: All inputs, outputs, weights are n-dimensional arrays (like NumPy arrays)
- **`torch.nn`**: building blocks (layers) that are combined into models -> Architecture
- **Weights**: the learned parameters, stored separately (`.pt`/`.pth`, `.safetensors`) and loaded into the architecture
- **Eager execution**: code runs line by line -> tensors can be printed and inspected at any time
- **Autograd**: computes gradients automatically -> needed for *training*

To use PyTorch based tools like *Demucs, torchaudio, BeatNet*, we only need **inference** (applying a trained model), no training or transfer task.

"Older" libraries/packages like *spleeter, essentia, CREPE* are based on TensorFlow! 



### Installation 
    conda activate mirml                      # Your course environment
    conda install -c conda-forge pytorch      # CPU or NVIDIA build is chosen automatically
    python -m pip install demucs              # not on conda-forge -> use pip inside the active env instead

- PyTorch is large (several hundred MB up to GBs with GPU support)
- On first use, Demucs downloads its weights once from Hugging Face. The warning
  "unauthenticated requests to the HF Hub" can be ignored (no account/token needed)

### Pre-trained Models

- *htdemucs*: first version of Hybrid Transformer Demucs. Trained on MusDB + 800 songs. Default model.
- *htdemucs_ft*: fine-tuned version of htdemucs, separation will take 4 times more time but might be a bit better. Same training set as htdemucs.
- *htdemucs_6s*: 6 sources version of htdemucs, with piano and guitar being added as sources. Note that the piano source is not working great at the moment.
- *hdemucs_mmi*: Hybrid Demucs v3, retrained on MusDB + 800 songs.
- *mdx*: trained only on MusDB HQ, winning model on track A at the MDX challenge.
- *mdx_extra*: trained with extra training data (including MusDB test set), ranked 2nd on the track B of the MDX challenge.
- *mdx_q*, mdx_extra_q: quantized version of the previous models. Smaller download and storage but quality can be slightly worse.
- *SIG*: where SIG is a single model from the model zoo.

--> Stem separation models are usually compared with the signal-to-distortion ratio (SDR; Vincent et al., 2006) implemented by *museval* as the evaluation metric

In [ ]:
from demucs.api import Separator, list_models
import torch

separator = Separator(model="htdemucs", progress=True)   # loads architecture + weights of selected model (download on first use)
model = separator.model                                    # a regular PyTorch framework (torch.nn.Module)

print("sources:", model.sources)                           # always estimated together
print("sample rate:", separator.samplerate, "| channels:", separator.audio_channels)
print("parameters:", f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f} M")
print("training mode:", model.training)                    # False = inference mode
print("available models:", sorted(list_models()["bag"]))

In [ ]:
import time
t0 = time.time()

TRACK =    # select a track of your choice as example 

mix, stems = separator.separate_audio_file(PREVIEW_DIR / f"{TRACK}.mp3")   # mix: resampled to 44.1 kHz stereo
print(f"{time.time() - t0:.0f} s for {mix.shape[1] / separator.samplerate:.0f} s of audio")
print("mix:", tuple(mix.shape), {name: tuple(s.shape) for name, s in stems.items()})   # (channels, samples)

# The stems are estimates: their sum does not reproduce the mix exactly!
residual = mix - sum(stems.values())
print(f"residual (mix − sum of stems): {20 * torch.log10(residual.std() / mix.std()).item():.1f} dB re mix")

### How HT Demucs works (Rouard, Massa & Défossez, 2023)

--> "*[...] an hybrid temporal/spectral bi-U-Net based on Hybrid Demucs, where the innermost layers are replaced by a cross-domain Transformer Encoder, using self-attention within one domain, and cross-attention across domains.*"

- **Hybrid approach**: one branch works on the waveform, one on the spectrogram; a Transformer connects both
- Separates **all 4 sources at once** (drums, bass, other, vocals).
- Audio is processed in **segments of ≤ 7.8 s** with overlap and then joined
- **Training data**: MUSDB18-HQ (Stems from 150 songs, mostly pop/rock) + 800 extra songs (unknown) -> "diverse music genres"
- **Augmentation**: time shift, channel swap, sign flip, gain, remixing stems across songs, pitch/tempo change
  -> **no** filters, EQ or reverb augmentation is represented
- Consequence for us: EDM-specific production (filter sweeps, vocal chops, formant-like synth leads)
  might be **outside the training distribution** -> My own guess


In [ ]:
'''
Comparison: Demucs stems vs. HPSS (harmonic–percussive source separation, librosa)
HPSS uses median filtering of the spectrogram: along time → sustained (harmonic) parts,
along frequency → broadband transients (percussive). No training, no model – but also no notion of "voice".
'''
TRACK =              # <- ID of selected track
EXCERPT = (0, 20)            # seconds within the preview
MARGIN = 2.0                  # HPSS margin: 1 = every bin goes to H or P; > 1 = stricter, remainder → residual
KERNEL = 31                   # median filter length (frames / frequency bins), librosa default
SR = 22050                    # make sure to select the matching sample rate you used to sxtract the stems

mix, _ = librosa.load(PREVIEW_DIR / f"{TRACK}.mp3", sr=SR)
drums, _ = sf.read(STEM_DIR / f"{TRACK}_drums.flac", dtype="float32")
vocals, _ = sf.read(STEM_DIR / f"{TRACK}_vocals.flac", dtype="float32")
a, b = (int(t * SR) for t in EXCERPT)
mix, drums, vocals = mix[a:b], drums[a:b], vocals[a:b]

harm, perc = librosa.effects.hpss(mix, kernel_size=KERNEL, margin=MARGIN)
versions = {"Demucs: drums": drums,
            "Demucs: mix − drums (bass, other, vocals)": mix - drums,
            "Demucs: vocals": vocals,
            "HPSS: percussive": perc,
            "HPSS: harmonic": harm,
            "HPSS: residual (mix − H − P)": mix - harm - perc}

# Spectrograms on a common dB scale (0 dB = loudest bin of the mix) → levels are comparable
ref = np.abs(librosa.stft(mix)).max()
fig, axes = plt.subplots(2, 3, figsize=(13, 6.5), sharex=True, sharey=True, constrained_layout=True)
for ax, (name, x) in zip(axes.ravel(), versions.items()):
    S_db = librosa.amplitude_to_db(np.abs(librosa.stft(x)), ref=ref)
    img = librosa.display.specshow(S_db, sr=SR, x_axis="time", y_axis="log", vmin=-80, vmax=0, ax=ax)
    share_db = 10 * np.log10(np.mean(x ** 2) / np.mean(mix ** 2))        # energy relative to the mix
    ax.set(title=f"{name}\n{share_db:+.1f} dB re mix", xlabel="", ylabel="")
fig.colorbar(img, ax=axes, format="%+.0f dB", shrink=0.8)
plt.show()

# Listen (real level; clipped to ±1 for playback)
print("mix"); display(Audio(np.clip(mix, -1, 1), rate=SR, normalize=False))
for name, x in versions.items():
    print(name)
    display(Audio(np.clip(x, -1, 1), rate=SR, normalize=False))

In [ ]:
# normalize=False keeps the real level → leakage stays as quiet as it really is
# (clip to ±1: decoded MP3s and stems can slightly exceed full scale)
for name, s in stems.items():
    print(name)
    display(Audio(np.clip(s.mean(0).numpy()[: 20 * separator.samplerate], -1, 1),
                  rate=separator.samplerate, normalize=False))

### Separator: relevant parameters

| Parameter | Default | Effect |
|---|---|---|
| `model` | `htdemucs` | `htdemucs_ft`: 4 specialised models, ~4× slower, slightly better. `htdemucs_6s`: + guitar, piano (piano weak according to README) |
| `device` | GPU if available | `"cpu"` / `"cuda"` |
| `shifts` | 1 | separates *n* times with a random offset (0–0.5 s) and averages → fewer artefacts, *n* times slower (README: only with GPU). **Random** → set `random.seed(...)` for reproducible stems |
| `overlap` | 0.25 | overlap of the segments; 0.1 is slightly faster |
| `split`, `segment` | True, ≤ 7.8 s | segment length; smaller = less memory |
| `jobs` | 0 | parallel workers on CPU → faster, but multiplies RAM |
| `progress`, `callback` | False, None | progress bar / own function per segment |

- `update_parameter(...)` changes settings without reloading the model
- `separate_tensor(wav, sr)` for audio that is already loaded (float32, shape channels × samples)
- Unlike the command line tool, the API does **not** rescale stems against clipping

### When does a track "contain vocals"?

There is no fixed definition at track level:
- **Singing voice detection** (MIR) decides frame by frame whether a voice is present
  (e.g. Lehner, Schlüter & Widmer 2018). A track label has to be derived from that.
- **Spotify "instrumentalness"**: "Ooh" and "aah" sounds count as instrumental, rap/spoken word as vocal;
  values > 0.5 are intended to represent instrumental tracks.
- **Electronic music is harder**: on EDM, singing voice detection reaches F1 0.83 vs. 0.94 on pop;
  source separation helped by > 20 % on pop but < 2 % on EDM (Romero-Arenas et al. 2022).

Possible criteria → how to measure them → threshold:
| Criterion | Measure | Threshold |
|---|---|---|
| amount | share of active frames in the preview | e.g. ≥ 5 % |
| level (foreground/background) | vocal stem relative to mix (dB) | e.g. > −15 dB active, > −6 dB foreground |
| continuity (phrase vs. one-shot) | length of connected active segments | e.g. longest segment ≥ 2 s |
| use (melodic vs. percussive) | onsets per second in the vocal stem | many short onsets = chops |

In [ ]:
# Vocal stem level relative to the mix over time, with candidate thresholds
# -> show for one track where (and how long) the vocal stem is clearly audible in the mix
TRACK =      # <- select a track here

# 1. Load both signals at the same sample rate (22.05 kHz, mono)
mix_m, _ = librosa.load(PREVIEW_DIR / f"{TRACK}.mp3", sr=SR)                 # original preview, resampled + mono
voc, _ = sf.read(STEM_DIR / f"{TRACK}_vocals.flac", dtype="float32")         # Demucs vocal stem (already 22.05 kHz mono)

# 2. Cut both to the same length (they can differ by a few samples after resampling)
n = min(len(mix_m), len(voc))
mix_m, voc = mix_m[:n], voc[:n]

# 3. Level per frame: RMS over short windows (2048 samples ≈ 93 ms, every 512 samples ≈ 23 ms)
#    librosa returns shape (1, frames) → [0] takes the single row
rms_mix = librosa.feature.rms(y=mix_m)[0]
rms_voc = librosa.feature.rms(y=voc)[0]

# 4. Vocal level relative to the mix in dB: 0 dB = as loud as the mix, −20 dB = 20 dB quieter
#    (+1e-10 avoids log(0) in silent frames)
rel_db = 20 * np.log10((rms_voc + 1e-10) / (rms_mix + 1e-10))

# 5. Time (s) of each frame for the x-axis
t = librosa.times_like(rel_db, sr=SR)

# Plot vocal level over time
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t, rel_db, lw=0.7)
for thr, ls in [(-6, "-"), (-15, "--"), (-25, ":")]:
    ax.axhline(thr, color="k", ls=ls, lw=0.8, label=f"{thr} dB")
ax.set(xlabel="time in preview (s)", ylabel="vocal stem re mix (dB)", ylim=(-50, 5), title=f"{TRACK}: vocal level over time")
ax.legend(loc="lower right", fontsize=7)
plt.show()